In [ ]:

from transformers import MarianTokenizer, MarianMTModel
from datasets import load_metric
from comet import download_model, load_from_checkpoint
import torch
import numpy as np
import time
import pandas as pd

/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# load dataset (cleaned)

In [2]:
# Load preprocessed dataset (created in previous step)
df = pd.read_csv("OpenSubtitles_en-fr_clean.csv")

df = df.head(300).reset_index(drop=True)

src_texts = df["src"].tolist()
tgt_texts = df["tgt"].tolist()

print(f"Loaded first {len(df):,} cleaned sentence pairs for evaluation.")


Loaded first 300 cleaned sentence pairs for evaluation.


In [3]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to("cuda" if torch.cuda.is_available() else "cpu")

print(f"Model loaded: {model_name}")


/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Model loaded: Helsinki-NLP/opus-mt-en-fr


In [4]:
preds, latencies = [], []

for text in src_texts:
    start = time.time()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    output = model.generate(**inputs)
    end = time.time()

    pred = tokenizer.decode(output[0], skip_special_tokens=True)
    preds.append(pred)
    latencies.append(end - start)

avg_latency = np.mean(latencies)
print(f"Average latency per sentence: {avg_latency:.3f} seconds")


Average latency per sentence: 0.351 seconds


In [5]:
metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

print(f"BLEU: {bleu['score']:.2f}")
print(f"chrF: {chrf['score']:.2f}")


/var/folders/00/hlc640bs3hx3jf_rqtdxfpkc0000gn/T/ipykernel_20861/2869170210.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric_bleu = load_metric("sacrebleu")


BLEU: 23.30
chrF: 53.11


In [ ]:
import numpy as np
from comet import download_model, load_from_checkpoint
import torch

torch.backends.mps.is_available = lambda: False

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]

comet_score = comet_model.predict(
    data,
    batch_size=4,
    gpus=0,
    num_workers=0
)

print(f"COMET mean score: {np.mean(comet_score['system_score']):.4f}")

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 100342.20it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Predicting DataLoader 0: 100%|██████████| 75/75 [00:34<00:00,  2.17it/s]

COMET mean score: 0.7577


In [9]:
input_lens = [len(tokenizer.encode(s)) for s in src_texts]
output_lens = [len(tokenizer.encode(p)) for p in preds]
avg_prop = np.mean(np.array(output_lens) / np.array(input_lens))

print(f"Average Proportion (AP): {avg_prop:.3f}")
print(f"Average Lagging (AL): ~0 (offline model)")
print(f"Differentiable AL (DAL): ~0 (offline model)")


Average Proportion (AP): 1.693
Average Lagging (AL): ~0 (offline model)
Differentiable AL (DAL): ~0 (offline model)


In [10]:
print("\n===== Baseline Evaluation Summary =====")
print(f"BLEU: {bleu['score']:.2f}")
print(f"chrF: {chrf['score']:.2f}")
print(f"COMET: {np.mean(comet_score['system_score']):.4f}")
print(f"Latency (sec/sentence): {avg_latency:.3f}")
print(f"AP: {avg_prop:.3f} | AL ≈ 0 | DAL ≈ 0")
print("=======================================")



===== Baseline Evaluation Summary =====
BLEU: 23.30
chrF: 53.11
COMET: 0.7577
Latency (sec/sentence): 0.351
AP: 1.693 | AL ≈ 0 | DAL ≈ 0


AL and DAL are only meaningful in simultaneous translation or real-time decoding experiments.
This baseline model processes the entire input sentence before producing any output. So it’s completely normal that both metrics are close to 0.